In [ ]:
import numpy as np
import random

# ── 1. 데이터 ───────────────────────────────────────────────────
CROPS    = ['양상추','토마토','대파','오이','고추','시금치','무','배추','옥수수']
N_CROP, N_PERIOD = 9, 24
area_per = np.array([0.10,0.30,0.15,0.40,0.25,0.05,0.20,0.20,0.20])
grow_O   = np.array([4,8,6,5,10,3,6,5,6])
grow_H   = np.array([3,6,5,4,8,2,5,4,5])

price_O_monthly = np.array([
    [2500,2500,2500,2500,2500,0,0,0,0,0,2200,2200],
    [0,0,0,0,15000,18000,18000,15000,12000,0,0,0],
    [0,0,4000,4000,4500,5000,5000,5500,5000,4500,0,0],
    [0,0,0,0,11000,11000,13000,15000,13000,11000,0,0],
    [0,0,0,0,0,0,10000,12000,12000,10000,0,0],
    [800,800,600,600,0,0,0,0,700,700,800,800],
    [0,0,0,0,0,0,0,0,4500,4800,4800,4800],
    [0,0,0,0,0,0,0,0,3200,3200,3600,3600],
    [0,0,0,0,0,0,3000,3500,3000,0,0,0],
])
price_H_monthly = np.array([
    [3500,3500,3200,3200,3200,3000,3000,3500,3500,3200,3200,3200],
    [0,0,25000,25000,21000,25000,25000,21000,18000,20000,22000,22000],
    [6000,6000,6000,6000,6500,7000,7000,8000,7000,6500,7000,7000],
    [0,0,15000,15000,15000,16000,18000,21000,18000,16000,18000,18000],
    [0,0,15000,15000,15000,15000,14000,17000,17000,15000,18000,18000],
    [1200,1200,1000,1000,1000,1000,1000,1200,1000,1000,1200,1200],
    [6500,6500,6000,6000,6000,6000,6000,6500,6000,6500,6500,6500],
    [5500,5500,5000,5000,5000,5000,5000,5500,4800,4800,5000,5000],
    [0,0,0,0,4500,4500,4200,5000,4200,4500,0,0],
])

AREA_O, AREA_H = 1200.0, 300.0
MAX_O, MAX_H   = 300.0, 75.0
MIN_O, MIN_H   = 60.0, 15.0

valid_O = np.zeros((N_CROP,N_PERIOD), dtype=bool)
valid_H = np.zeros((N_CROP,N_PERIOD), dtype=bool)
rev_O   = np.zeros((N_CROP,N_PERIOD))
rev_H   = np.zeros((N_CROP,N_PERIOD))
for i in range(N_CROP):
    for t in range(N_PERIOD):
        ht = t+grow_O[i]-1
        if ht<N_PERIOD and price_O_monthly[i,ht//2]>0:
            valid_O[i,t]=True; rev_O[i,t]=price_O_monthly[i,ht//2]/area_per[i]
        ht = t+grow_H[i]-1
        if ht<N_PERIOD and price_H_monthly[i,ht//2]>0:
            valid_H[i,t]=True; rev_H[i,t]=price_H_monthly[i,ht//2]/area_per[i]

# 후보 수집 (gene pool)
# 튜플: (단위수익, 성장기간, 작물번호, 파종시기, 농지종류)
candidates = []
for t in range(N_PERIOD):
    for i in range(N_CROP):
        if valid_O[i,t]: candidates.append((rev_O[i,t], grow_O[i], i, t, 'O'))
        if valid_H[i,t]: candidates.append((rev_H[i,t], grow_H[i], i, t, 'H'))

In [ ]:
# =====================================================
# ★ 여기를 고치세요! ★
# 정렬 알고리즘만 바꾸면 됩니다. key는 동일.
# key: 단위수익 ↓  성장기간 ↑  작물번호 ↑
# =====================================================

def quicksort(arr):
    if len(arr) <= 1:
        return arr
    pivot = arr[len(arr) // 2]
    pk = (-pivot[0], pivot[1], pivot[2])
    left  = [x for x in arr if (-x[0], x[1], x[2]) < pk]
    mid   = [x for x in arr if (-x[0], x[1], x[2]) == pk]
    right = [x for x in arr if (-x[0], x[1], x[2]) > pk]
    return quicksort(left) + mid + quicksort(right)

def insertion_sort(arr):
    a = arr[:]
    for i in range(1, len(a)):
        cur = a[i]
        ck = (-cur[0], cur[1], cur[2])
        j = i - 1
        while j >= 0 and (-a[j][0], a[j][1], a[j][2]) > ck:
            a[j + 1] = a[j]
            j -= 1
        a[j + 1] = cur
    return a

# [방식A] 퀵소트
candidates = quicksort(candidates)

# [방식B] 인서션 소트 (아래 줄로 교체)
# candidates = insertion_sort(candidates)
# =====================================================

In [ ]:
# ── 3. GA 핵심 함수 ─────────────────────────────────────────────

def decode(chromosome):
    """chromosome (0/1 리스트) → xO, xH 배열 (그리디 방식으로 면적 배정)"""
    xO = np.zeros((N_CROP,N_PERIOD)); xH = np.zeros((N_CROP,N_PERIOD))
    oO = np.zeros(N_PERIOD);          oH = np.zeros(N_PERIOD)
    for gene, (rv, gt, i, t, land) in zip(chromosome, candidates):
        if not gene: continue
        et = t + gt
        if et > N_PERIOD: continue
        if land == 'O':
            av = AREA_O - oO[t:et].max(); al = min(MAX_O, av)
            if al >= MIN_O: oO[t:et]+=al; xO[i,t]=al
        else:
            av = AREA_H - oH[t:et].max(); al = min(MAX_H, av)
            if al >= MIN_H: oH[t:et]+=al; xH[i,t]=al
    return xO, xH

def fitness(chromosome):
    xO, xH = decode(chromosome)
    return (rev_O * xO + rev_H * xH).sum()

def greedy_chromosome():
    """sort 순서대로 그리디 배정 → 초기 chromosome"""
    chrom = [0] * len(candidates)
    oO = np.zeros(N_PERIOD); oH = np.zeros(N_PERIOD)
    for idx, (rv, gt, i, t, land) in enumerate(candidates):
        et = t + gt
        if et > N_PERIOD: continue
        if land == 'O':
            av = AREA_O - oO[t:et].max(); al = min(MAX_O, av)
            if al >= MIN_O: chrom[idx]=1; oO[t:et]+=al
        else:
            av = AREA_H - oH[t:et].max(); al = min(MAX_H, av)
            if al >= MIN_H: chrom[idx]=1; oH[t:et]+=al
    return chrom

def init_population(pop_size):
    """그리디 해 1개 + 랜덤 변형 (pop_size-1)개"""
    base = greedy_chromosome()
    pop  = [base]
    for _ in range(pop_size - 1):
        mutated = [1-g if random.random()<0.2 else g for g in base]
        pop.append(mutated)
    return pop

def crossover(p1, p2):
    pt = random.randint(1, len(p1)-1)
    return p1[:pt]+p2[pt:], p2[:pt]+p1[pt:]

def mutate(chrom, rate=0.05):
    return [1-g if random.random()<rate else g for g in chrom]

def select(pop, fits, k=3):
    idxs = random.sample(range(len(pop)), k)
    return pop[max(idxs, key=lambda i: fits[i])]

# ── 4. GA 실행 ───────────────────────────────────────────────────
import time

POP_SIZE      = 30
GENERATIONS   = 100
MUTATION_RATE = 0.05

random.seed(42)
pop  = init_population(POP_SIZE)
fits = [fitness(c) for c in pop]

t0 = time.perf_counter()
for gen in range(GENERATIONS):
    elite   = pop[max(range(POP_SIZE), key=lambda i: fits[i])]
    new_pop = [elite]
    while len(new_pop) < POP_SIZE:
        c1, c2 = crossover(select(pop, fits), select(pop, fits))
        new_pop += [mutate(c1, MUTATION_RATE), mutate(c2, MUTATION_RATE)]
    pop  = new_pop[:POP_SIZE]
    fits = [fitness(c) for c in pop]
elapsed = time.perf_counter() - t0

best_chrom = pop[max(range(POP_SIZE), key=lambda i: fits[i])]
xO, xH    = decode(best_chrom)
crop_rev  = [(rev_O[i]*xO[i]+rev_H[i]*xH[i]).sum() for i in range(N_CROP)]
total_rev = sum(crop_rev)

# ── 5. 출력 ─────────────────────────────────────────────────────
print("=" * 52)
print(f"  GA 최적해: {total_rev/1e6:.4f} 백만원")
print(f"  세대: {GENERATIONS}  집단: {POP_SIZE}  /  소요시간: {elapsed*1000:.1f} ms")
print("=" * 52)
print(f"\n{'작물':<6} {'수익(백만원)':>12} {'노지(m2)':>10} {'하우스(m2)':>10}")
print("-" * 42)
for i in range(N_CROP):
    print(f"{CROPS[i]:<6} {crop_rev[i]/1e6:>12.3f} {xO[i].sum():>10.1f} {xH[i].sum():>10.1f}")
print("-" * 42)
print(f"{'합계':<6} {total_rev/1e6:>12.4f} {xO.sum():>10.1f} {xH.sum():>10.1f}")